# 물리적 리스크 PRF 세부 증가율 연도별 매핑

이 노트북은 기존 `물리적리스크_PRF_세부증가율_매핑.xlsx` 파일을 기반으로, 사업장별·기업별 PRF를 0~40년까지 1년 단위로 계산하는 코드임.

기존 파일에는 `PRF_10Y`, `PRF_20Y`, `PRF_30Y`만 있었지만, 517 시트의 연도별 시뮬레이션과 연결하려면 매년의 PRF 값이 필요함.  
따라서 본 노트북에서는 다음 4개 시트를 추가 생성함.

- `사업장별 PRF_연도별`: 사업장 × 연도 long-format 패널
- `기업별 PRF_연도별`: 기업 × 연도 long-format 패널
- `사업장별 PRF_wide40`: 사업장별 PRF_0Y~PRF_40Y wide-format
- `기업별 PRF_wide40`: 기업별 PRF_0Y~PRF_40Y wide-format

계산 방식은 기존 PRF 구조를 유지함.

```text
PRF(t) = drr_factor(0) × (1 + g_drr)^t
       + B_factor(0) × (1 + g_B)^t
       - 1
```

여기서 `B`는 하천범람위험과 해안침수위험 중 더 큰 지배 홍수위험임.


In [2]:
# 필요한 패키지 불러오기임
from pathlib import Path
import numpy as np
import pandas as pd

# 입력/출력 파일 경로임
# 같은 폴더에 파일을 두고 실행하면 됨
input_path = Path("/Users/dapanman/Downloads/물리적리스크_PRF_세부증가율_매핑.xlsx")
output_path = Path("/Users/dapanman/Downloads/물리적리스크_PRF_연도별_매핑.xlsx")


print(input_path)
print(output_path)


/Users/dapanman/Downloads/물리적리스크_PRF_세부증가율_매핑.xlsx
/Users/dapanman/Downloads/물리적리스크_PRF_연도별_매핑.xlsx


## 1. 기존 사업장별 PRF 데이터 불러오기

기준 시트는 `사업장별 PRF`임.  
이 시트에는 사업장별 `drr 팩터`, `B 팩터`, `PRF`, `drr 증가율`, `B 증가율` 등이 이미 들어 있음.


In [3]:
site_df = pd.read_excel(input_path, sheet_name="사업장별 PRF")

# 완전 빈 행 제거함
site_df = site_df[site_df["기업명"].notna()].copy()

print("사업장 수:", len(site_df))
display(site_df.head())
display(site_df.columns)


사업장 수: 163


,기업명,사업장명,사업장 위치,면적(km²),위도,경도,rfr_score,cfr_score,drr_score,B 점수,...,drr 증가율,rfr 증가율,cfr 증가율,지배 홍수위험,B 증가율,PRF_10Y,PRF_20Y,PRF_30Y,PRF 유효CAGR(30Y),매핑 근거/주의
0,삼성전자,수원사업장,경기도 수원시 영통구 삼성로 129(매탄동),1.30000,37.257643,127.053129,1.065214,2.072041,2.263827,2.072041,...,0.0,0.0025,0.007,CFR,0.0070,1.162294,1.242988,1.329512,0.006734,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
1,삼성전자,서초사업장,서울특별시 서초구 서초대로74길 11(서초동),0.01340,37.496633,127.026914,0.562331,0.000000,1.965401,0.562331,...,0.0,0.0025,0.000,RFR,0.0025,0.854798,0.878704,0.903215,0.002762,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
2,삼성전자,우면사업장,서울특별시 서초구 성촌길 33(우면동),0.03720,37.465025,127.022762,0.562331,0.000000,1.965401,0.562331,...,0.0,0.0025,0.000,RFR,0.0025,0.854798,0.878704,0.903215,0.002762,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
3,삼성전자,기흥사업장,경기도 용인시 기흥구 삼성로 1(농서동),0.38430,37.226944,127.087764,1.065214,2.072041,2.263827,2.072041,...,0.0,0.0025,0.007,CFR,0.0070,1.162294,1.242988,1.329512,0.006734,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
4,삼성전자,화성사업장,경기도 화성시 삼성전자로 1(반월동),0.91933,37.223465,127.063384,1.065214,2.072041,2.263827,2.072041,...,0.0,0.0025,0.007,CFR,0.0070,1.162294,1.242988,1.329512,0.006734,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."


Index(['기업명', '사업장명', '사업장 위치', '면적(km²)', '위도', '경도', 'rfr_score',
       'cfr_score', 'drr_score', 'B 점수', 'drr 백분위', 'B 백분위', 'drr 팩터', 'B 팩터',
       'PRF', '유역/권역(근사)', 'drr 증가율', 'rfr 증가율', 'cfr 증가율', '지배 홍수위험', 'B 증가율',
       'PRF_10Y', 'PRF_20Y', 'PRF_30Y', 'PRF 유효CAGR(30Y)', '매핑 근거/주의'],
      dtype='object')

## 2. 연도 범위 설정

517 시트의 장기 시뮬레이션과 맞추기 위해 0년차부터 40년차까지 1년 단위로 계산함.  
`t=0`은 기존 엑셀의 기준시점 PRF임.


In [4]:
years = list(range(0, 41))
print(years[:5], "...", years[-5:])


[0, 1, 2, 3, 4] ... [36, 37, 38, 39, 40]


## 3. 사업장별 연도별 PRF 계산

기존 계산식은 다음과 같음.

```text
PRF = drr_factor + B_factor - 1
```

이를 시간축으로 확장하면 다음과 같음.

```text
drr_factor(t) = drr_factor(0) × (1 + g_drr)^t
B_factor(t)   = B_factor(0) × (1 + g_B)^t
PRF(t)        = drr_factor(t) + B_factor(t) - 1
```

`g_B`는 각 사업장에서 지배 홍수위험이 RFR이면 `rfr 증가율`, CFR이면 `cfr 증가율`이 들어간 값임.  
기존 파일에는 이미 `B 증가율` 컬럼으로 들어 있음.


In [5]:
site_long_rows = []
site_wide = site_df[[
    "기업명", "사업장명", "사업장 위치", "면적(km²)",
    "drr 팩터", "B 팩터", "PRF",
    "유역/권역(근사)", "drr 증가율", "rfr 증가율", "cfr 증가율",
    "지배 홍수위험", "B 증가율"
]].copy()

for y in years:
    drr_t = site_df["drr 팩터"] * (1 + site_df["drr 증가율"]) ** y
    b_t = site_df["B 팩터"] * (1 + site_df["B 증가율"]) ** y
    prf_t = drr_t + b_t - 1

    # wide-format용 컬럼 추가함
    site_wide[f"PRF_{y}Y"] = prf_t

    # long-format용 데이터 생성함
    tmp = site_df[[
        "기업명", "사업장명", "사업장 위치", "면적(km²)",
        "유역/권역(근사)", "지배 홍수위험",
        "drr 증가율", "B 증가율"
    ]].copy()
    tmp["연도"] = y
    tmp["drr_factor_t"] = drr_t
    tmp["B_factor_t"] = b_t
    tmp["PRF_t"] = prf_t

    site_long_rows.append(tmp)

site_long = pd.concat(site_long_rows, ignore_index=True)

# 전년 대비 PRF 증가율 계산함
site_long = site_long.sort_values(["기업명", "사업장명", "연도"]).reset_index(drop=True)
site_long["PRF_YoY"] = site_long.groupby(["기업명", "사업장명"])["PRF_t"].pct_change()

# 보기 좋은 컬럼 순서로 정리함
site_long = site_long[[
    "기업명", "사업장명", "사업장 위치", "면적(km²)", "연도",
    "유역/권역(근사)", "지배 홍수위험",
    "drr 증가율", "B 증가율",
    "drr_factor_t", "B_factor_t", "PRF_t", "PRF_YoY"
]]

print("사업장별 연도별 행 수:", len(site_long))
display(site_long.head(10))
display(site_wide.head())


사업장별 연도별 행 수: 6683


/var/folders/h6/c563qbpd3dz735x4bqtwkzpr0000gn/T/ipykernel_77417/619971679.py:34: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  site_long["PRF_YoY"] = site_long.groupby(["기업명", "사업장명"])["PRF_t"].pct_change()


,기업명,사업장명,사업장 위치,면적(km²),연도,유역/권역(근사),지배 홍수위험,drr 증가율,B 증가율,drr_factor_t,B_factor_t,PRF_t,PRF_YoY
0,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,0,영산강/섬진강권,RFR,0.0027,0.0045,1.107407,1.120370,1.227778,NaN
1,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,1,영산강/섬진강권,RFR,0.0027,0.0045,1.110397,1.125412,1.235809,0.006542
2,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,2,영산강/섬진강권,RFR,0.0027,0.0045,1.113395,1.130476,1.243872,0.006524
3,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,3,영산강/섬진강권,RFR,0.0027,0.0045,1.116402,1.135564,1.251965,0.006507
4,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,4,영산강/섬진강권,RFR,0.0027,0.0045,1.119416,1.140674,1.260090,0.006489
5,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,5,영산강/섬진강권,RFR,0.0027,0.0045,1.122438,1.145807,1.268245,0.006472
6,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,6,영산강/섬진강권,RFR,0.0027,0.0045,1.125469,1.150963,1.276432,0.006455
7,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,7,영산강/섬진강권,RFR,0.0027,0.0045,1.128508,1.156142,1.284650,0.006438
8,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,8,영산강/섬진강권,RFR,0.0027,0.0045,1.131555,1.161345,1.292899,0.006422
9,HD한국조선해양,영암조선소,전라남도 영암군 삼호읍 대불로 93,2.1,9,영산강/섬진강권,RFR,0.0027,0.0045,1.134610,1.166571,1.301181,0.006405


,기업명,사업장명,사업장 위치,면적(km²),drr 팩터,B 팩터,PRF,유역/권역(근사),drr 증가율,rfr 증가율,...,PRF_31Y,PRF_32Y,PRF_33Y,PRF_34Y,PRF_35Y,PRF_36Y,PRF_37Y,PRF_38Y,PRF_39Y,PRF_40Y
0,삼성전자,수원사업장,경기도 수원시 영통구 삼성로 129(매탄동),1.30000,1.045370,1.041667,1.087037,한강권/수도권,0.0,0.0025,...,1.338501,1.347553,1.356668,1.365847,1.375090,1.384399,1.393772,1.403211,1.412715,1.422287
1,삼성전자,서초사업장,서울특별시 서초구 서초대로74길 11(서초동),0.01340,0.909259,0.922222,0.831481,한강권/수도권,0.0,0.0025,...,0.905700,0.908191,0.910688,0.913192,0.915702,0.918218,0.920740,0.923269,0.925804,0.928345
2,삼성전자,우면사업장,서울특별시 서초구 성촌길 33(우면동),0.03720,0.909259,0.922222,0.831481,한강권/수도권,0.0,0.0025,...,0.905700,0.908191,0.910688,0.913192,0.915702,0.918218,0.920740,0.923269,0.925804,0.928345
3,삼성전자,기흥사업장,경기도 용인시 기흥구 삼성로 1(농서동),0.38430,1.045370,1.041667,1.087037,한강권/수도권,0.0,0.0025,...,1.338501,1.347553,1.356668,1.365847,1.375090,1.384399,1.393772,1.403211,1.412715,1.422287
4,삼성전자,화성사업장,경기도 화성시 삼성전자로 1(반월동),0.91933,1.045370,1.041667,1.087037,한강권/수도권,0.0,0.0025,...,1.338501,1.347553,1.356668,1.365847,1.375090,1.384399,1.393772,1.403211,1.412715,1.422287


## 4. 기업별 면적가중 PRF 계산

사업장별 PRF를 기업 단위로 묶을 때는 단순평균이 아니라 면적가중평균을 사용함.

```text
기업 PRF(t) = Σ[사업장 면적 × 사업장 PRF(t)] / Σ[사업장 면적]
```

면적이 큰 사업장이 기업 전체 물리적 리스크에 더 큰 영향을 준다고 보기 때문임.


In [6]:
def weighted_avg(group, value_col, weight_col="면적(km²)"):
    values = pd.to_numeric(group[value_col], errors="coerce")
    weights = pd.to_numeric(group[weight_col], errors="coerce")
    mask = values.notna() & weights.notna() & (weights > 0)
    if mask.sum() == 0:
        return values.mean()
    return np.average(values[mask], weights=weights[mask])

company_rows = []

for (company, year), g in site_long.groupby(["기업명", "연도"]):
    prf_t = weighted_avg(g, "PRF_t")
    drr_t = weighted_avg(g, "drr_factor_t")
    b_t = weighted_avg(g, "B_factor_t")
    g_drr = weighted_avg(g, "drr 증가율")
    g_b = weighted_avg(g, "B 증가율")
    total_area = pd.to_numeric(g["면적(km²)"], errors="coerce").sum()
    n_sites = g["사업장명"].nunique()

    # 지배 홍수위험은 최빈값으로 요약함
    mode_vals = g["지배 홍수위험"].mode()
    dom_mode = mode_vals.iloc[0] if len(mode_vals) > 0 else None

    company_rows.append({
        "기업명": company,
        "연도": year,
        "사업장 수": n_sites,
        "총 면적(km²)": total_area,
        "PRF_t": prf_t,
        "drr_factor_t": drr_t,
        "B_factor_t": b_t,
        "면적가중 drr 증가율": g_drr,
        "면적가중 B 증가율": g_b,
        "지배 홍수위험(최빈)": dom_mode,
    })

company_long = pd.DataFrame(company_rows).sort_values(["기업명", "연도"]).reset_index(drop=True)

# 전년 대비 증가율과 누적 CAGR 계산함
company_long["PRF_YoY"] = company_long.groupby("기업명")["PRF_t"].pct_change()
company_long["PRF_0Y"] = company_long.groupby("기업명")["PRF_t"].transform("first")
company_long["PRF 유효CAGR(누적)"] = np.where(
    company_long["연도"] > 0,
    (company_long["PRF_t"] / company_long["PRF_0Y"]) ** (1 / company_long["연도"]) - 1,
    np.nan
)

company_long = company_long[[
    "기업명", "연도", "사업장 수", "총 면적(km²)",
    "PRF_t", "drr_factor_t", "B_factor_t",
    "면적가중 drr 증가율", "면적가중 B 증가율",
    "PRF_YoY", "PRF 유효CAGR(누적)", "지배 홍수위험(최빈)"
]]

print("기업별 연도별 행 수:", len(company_long))
display(company_long.head(10))


기업별 연도별 행 수: 2050


/var/folders/h6/c563qbpd3dz735x4bqtwkzpr0000gn/T/ipykernel_77417/651778051.py:40: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  company_long["PRF_YoY"] = company_long.groupby("기업명")["PRF_t"].pct_change()


,기업명,연도,사업장 수,총 면적(km²),PRF_t,drr_factor_t,B_factor_t,면적가중 drr 증가율,면적가중 B 증가율,PRF_YoY,PRF 유효CAGR(누적),지배 홍수위험(최빈)
0,HD한국조선해양,0,2,2.731,1.253022,1.121527,1.131495,0.002076,0.0045,NaN,NaN,RFR
1,HD한국조선해양,1,2,2.731,1.260413,1.123826,1.136587,0.002076,0.0045,0.005898,0.005898,RFR
2,HD한국조선해양,2,2,2.731,1.267833,1.126132,1.141701,0.002076,0.0045,0.005887,0.005893,RFR
3,HD한국조선해양,3,2,2.731,1.275282,1.128443,1.146839,0.002076,0.0045,0.005876,0.005887,RFR
4,HD한국조선해양,4,2,2.731,1.282761,1.130761,1.152000,0.002076,0.0045,0.005864,0.005881,RFR
5,HD한국조선해양,5,2,2.731,1.290269,1.133085,1.157184,0.002076,0.0045,0.005853,0.005876,RFR
6,HD한국조선해양,6,2,2.731,1.297807,1.135416,1.162391,0.002076,0.0045,0.005842,0.005870,RFR
7,HD한국조선해양,7,2,2.731,1.305374,1.137752,1.167622,0.002076,0.0045,0.005831,0.005864,RFR
8,HD한국조선해양,8,2,2.731,1.312971,1.140095,1.172876,0.002076,0.0045,0.005820,0.005859,RFR
9,HD한국조선해양,9,2,2.731,1.320599,1.142444,1.178154,0.002076,0.0045,0.005809,0.005853,RFR


## 5. 기업별 wide-format 생성

wide-format은 517의 기업별 연도별 시뮬레이션에서 특정 연도의 PRF를 빠르게 가져오거나, 기업별 PRF trajectory를 가로로 비교할 때 유용함.


In [7]:
company_wide = company_long.pivot(index="기업명", columns="연도", values="PRF_t")
company_wide.columns = [f"PRF_{int(c)}Y" for c in company_wide.columns]
company_wide = company_wide.reset_index()

company_meta = company_long[company_long["연도"] == 0][[
    "기업명", "사업장 수", "총 면적(km²)", 
    "면적가중 drr 증가율", "면적가중 B 증가율", "지배 홍수위험(최빈)"
]]

company_wide = company_meta.merge(company_wide, on="기업명", how="left")

display(company_wide.head())


,기업명,사업장 수,총 면적(km²),면적가중 drr 증가율,면적가중 B 증가율,지배 홍수위험(최빈),PRF_0Y,PRF_1Y,PRF_2Y,PRF_3Y,...,PRF_31Y,PRF_32Y,PRF_33Y,PRF_34Y,PRF_35Y,PRF_36Y,PRF_37Y,PRF_38Y,PRF_39Y,PRF_40Y
0,HD한국조선해양,2,2.7310,0.002076,0.004500,RFR,1.253022,1.260413,1.267833,1.275282,...,1.496236,1.504588,1.512972,1.521390,1.529842,1.538327,1.546845,1.555398,1.563984,1.572604
1,HD현대미포,4,5.4490,0.000000,0.004500,RFR,1.337037,1.342295,1.347577,1.352883,...,1.511543,1.517586,1.523657,1.529755,1.535881,1.542034,1.548215,1.554423,1.560660,1.566924
2,HD현대일렉트릭,3,0.2981,0.000000,0.004547,RFR,1.332341,1.337637,1.342958,1.348303,...,1.508292,1.514392,1.520520,1.526676,1.532860,1.539072,1.545313,1.551583,1.557881,1.564207
3,HD현대중공업,1,4.7000,0.000000,0.004500,RFR,1.337037,1.342295,1.347577,1.352883,...,1.511543,1.517586,1.523657,1.529755,1.535881,1.542034,1.548215,1.554423,1.560660,1.566924
4,HL만도,3,0.3740,0.002450,0.006539,CFR,1.007759,1.017293,1.026910,1.036611,...,1.345613,1.358120,1.370738,1.383469,1.396314,1.409274,1.422349,1.435542,1.448853,1.462284


## 6. 설명 시트 생성

엑셀 파일 안에 연도별 확장 방식과 수식을 간단히 설명하는 시트를 함께 저장함.


In [8]:
annual_notes = pd.DataFrame([
    ["연도 범위", "0~40년, 1년 단위"],
    ["핵심 수정", "기존 PRF_10Y/20Y/30Y 스냅샷을 PRF_0Y~PRF_40Y 및 long-format 연도별 패널로 확장"],
    ["사업장별 PRF(t)", "drr_factor(0)*(1+g_drr)^t + B_factor(0)*(1+g_B)^t - 1"],
    ["B 증가율", "rfr_score >= cfr_score이면 g_rfr, cfr_score > rfr_score이면 g_cfr"],
    ["기업별 PRF(t)", "사업장별 PRF(t)의 면적가중평균"],
    ["drr 증가율", "금강권 0.77%, 영산강/섬진강권 0.27%, 기타 권역 0.00%"],
    ["rfr 증가율", "rfr_score 분위수 기준 0.25%, 0.35%, 0.45%"],
    ["cfr 증가율", "cfr_score=0이면 0.00%, 양수 구간 분위수 기준 0.70%, 1.00%, 1.16%"],
    ["주의", "주소 문자열 기반 근사 매핑임. 정밀 분석은 GIS 공간조인 필요"],
], columns=["항목", "내용"])

display(annual_notes)


,항목,내용
0,연도 범위,"0~40년, 1년 단위"
1,핵심 수정,기존 PRF_10Y/20Y/30Y 스냅샷을 PRF_0Y~PRF_40Y 및 long-...
2,사업장별 PRF(t),drr_factor(0)*(1+g_drr)^t + B_factor(0)*(1+g_B...
3,B 증가율,"rfr_score >= cfr_score이면 g_rfr, cfr_score > rf..."
4,기업별 PRF(t),사업장별 PRF(t)의 면적가중평균
5,drr 증가율,"금강권 0.77%, 영산강/섬진강권 0.27%, 기타 권역 0.00%"
6,rfr 증가율,"rfr_score 분위수 기준 0.25%, 0.35%, 0.45%"
7,cfr 증가율,"cfr_score=0이면 0.00%, 양수 구간 분위수 기준 0.70%, 1.00%..."
8,주의,주소 문자열 기반 근사 매핑임. 정밀 분석은 GIS 공간조인 필요


## 7. 엑셀 파일로 저장

기존 시트는 유지하고, 연도별 계산 시트 5개를 새로 추가함.

- `사업장별 PRF_연도별`
- `기업별 PRF_연도별`
- `사업장별 PRF_wide40`
- `기업별 PRF_wide40`
- `PRF_연도별_설명`


In [ ]:
# 기존 workbook에 새 시트를 덧붙여 저장함
# openpyxl 엔진 사용함
with pd.ExcelWriter(output_path, engine="openpyxl", mode="w") as writer:
    # 원본 시트 복사 저장함
    xls = pd.ExcelFile(input_path)
    for sheet in xls.sheet_names:
        original = pd.read_excel(input_path, sheet_name=sheet)
        original.to_excel(writer, sheet_name=sheet, index=False)

    # 신규 연도별 시트 저장함
    site_long.to_excel(writer, sheet_name="사업장별 PRF_연도별", index=False)
    company_long.to_excel(writer, sheet_name="기업별 PRF_연도별", index=False)
    site_wide.to_excel(writer, sheet_name="사업장별 PRF_wide40", index=False)
    company_wide.to_excel(writer, sheet_name="기업별 PRF_wide40", index=False)
    annual_notes.to_excel(writer, sheet_name="PRF_연도별_설명", index=False)

print("저장 완료:", output_path)


## 8. 간단 검증

사업장 수, 기업 수, 연도 수가 의도대로 생성됐는지 확인함.

예상 행 수는 다음과 같음.

```text
사업장별 연도별 행 수 = 사업장 수 × 41
기업별 연도별 행 수 = 기업 수 × 41
```


In [ ]:
print("사업장 수:", site_df.shape[0])
print("사업장별 연도별 행 수:", site_long.shape[0])
print("기업 수:", company_long["기업명"].nunique())
print("기업별 연도별 행 수:", company_long.shape[0])

print("사업장별 연도별 예상:", site_df.shape[0] * 41)
print("기업별 연도별 예상:", company_long["기업명"].nunique() * 41)

display(company_long[company_long["연도"].isin([0, 1, 10, 20, 30, 40])].head(20))
